# Prepare the MOM6 regional SSH files for SFINCS

**Author:** Faezeh Maghsoodifar, The University of Alabama  
**Case:** `gom12_ike_ERA5full.001` (Hurricane Ike, full ERA5 atmospheric forcing)

This standalone notebook finds the completed Texas and Mississippi regional SSH files from the successful MOM6 run, validates them, and copies them from Derecho scratch to a persistent `/glade/work` directory. It does **not** rebuild or rerun MOM6 and does **not** regenerate ERA5 forcing.

> The copied NetCDF files are the regional MOM6 water-level source products for the SFINCS workflow. If the selected SFINCS setup requires boundary-point time series or another schema, pass these files through that setup's MOM6-to-SFINCS/AutoCF conversion step.

## 1. Configuration

Edit only this cell if the case or destination changes. The destination is placed under `/glade/work`, not scratch, so the files are retained.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import hashlib
import json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

CASE_NAME = "gom12_ike_ERA5full.001"
CASE_DIR = Path("/glade/work/faezehmaghso/crocodile2026/croc_cases") / CASE_NAME
DEST_DIR = Path("/glade/work/faezehmaghso/crocodile2026/sfincs_input") / CASE_NAME
REGIONS = {"TX": "Texas coast", "MS": "Mississippi coast"}
JOIN_IF_NEEDED = True
OVERWRITE = False

assert CASE_DIR.is_dir(), f"Case directory not found: {CASE_DIR}"
print("Case:", CASE_DIR)
print("Persistent destination:", DEST_DIR)

## 2. Locate the Derecho run directory

This reads `RUNDIR` from the existing CESM case, so no variables from the original CrocoDash notebook are required.

In [ ]:
result = subprocess.run(
    [str(CASE_DIR / "xmlquery"), "--value", "RUNDIR"],
    cwd=CASE_DIR, check=True, capture_output=True, text=True,
)
RUNDIR = Path(result.stdout.strip())
assert RUNDIR.is_dir(), f"Run directory not found: {RUNDIR}"
print("RUNDIR:", RUNDIR)

## 3. Find or join the regional output

A filename ending exactly in `.nc` is the joined product. Names such as `.nc.0088` are parallel output pieces. The fallback join runs only if the completed file is missing.

In [ ]:
def completed_path(region):
    return RUNDIR / f"{CASE_NAME}.mom6.sfincs_{region}.nc"


def numeric_pieces(path):
    pieces = []
    for candidate in sorted(path.parent.glob(path.name + ".*")):
        if candidate.name.rsplit(".", 1)[-1].isdigit():
            pieces.append(candidate)
    return pieces


def join_if_missing(region):
    target = completed_path(region)
    if target.is_file():
        print(f"{region}: using completed file {target.name}")
        return target

    pieces = numeric_pieces(target)
    if not pieces:
        raise FileNotFoundError(f"No completed file or pieces found for {region}: {target}")
    if not JOIN_IF_NEEDED:
        raise FileNotFoundError(f"{target.name} is missing; found {len(pieces)} pieces")

    print(f"{region}: joining {len(pieces)} parallel pieces")
    opened = [xr.open_dataset(p) for p in pieces]
    try:
        combined = xr.combine_by_coords(opened, combine_attrs="override")
        temporary = target.with_name(target.name + ".joining")
        combined.to_netcdf(temporary, unlimited_dims=["time"] if "time" in combined.dims else None)
        combined.close()
        os.replace(temporary, target)
    finally:
        for ds in opened:
            ds.close()
    return target


source_files = {region: join_if_missing(region) for region in REGIONS}
source_files

## 4. Validate content and coverage

The expected field is `SSH_inst`, in meters. The successful Ike products should cover 2008-08-30 00:10 through 2008-09-20 00:00.

In [ ]:
def inspect_file(region, path):
    with xr.open_dataset(path) as ds:
        if "SSH_inst" not in ds:
            raise KeyError(f"{path.name} does not contain SSH_inst; variables: {list(ds.data_vars)}")
        ssh = ds["SSH_inst"]
        if "time" not in ssh.dims:
            raise ValueError(f"SSH_inst in {path.name} has no time dimension")
        first = str(ds.time.values[0])
        last = str(ds.time.values[-1])
        finite = int(np.isfinite(ssh.isel(time=0).values).sum())
        return {
            "region": region,
            "description": REGIONS[region],
            "file": path.name,
            "size_MB": round(path.stat().st_size / 1024**2, 2),
            "dimensions": dict(ssh.sizes),
            "first_time": first,
            "last_time": last,
            "finite_cells_at_first_time": finite,
            "units": ssh.attrs.get("units", "not recorded"),
        }


summary = [inspect_file(region, path) for region, path in source_files.items()]
pd.DataFrame(summary)

## 5. Copy the completed files to persistent storage

This protects the products from scratch cleanup. Existing files are not silently overwritten.

In [ ]:
DEST_DIR.mkdir(parents=True, exist_ok=True)
ready_files = {}

for region, source in source_files.items():
    destination = DEST_DIR / source.name
    if destination.exists() and not OVERWRITE:
        if destination.stat().st_size != source.stat().st_size:
            raise FileExistsError(
                f"Destination exists with a different size: {destination}. "
                "Inspect it or set OVERWRITE=True."
            )
        print(f"{region}: already copied; keeping {destination}")
    else:
        shutil.copy2(source, destination)
        print(f"{region}: copied to {destination}")
    ready_files[region] = destination

ready_files

## 6. Create a reproducibility manifest

The manifest records file size, SHA-256 checksum, variables, dimensions, and time coverage.

In [ ]:
def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()


manifest = {
    "author": "Faezeh Maghsoodifar",
    "case": CASE_NAME,
    "source_run_directory": str(RUNDIR),
    "description": "Regional MOM6 instantaneous sea-surface-height products for downstream SFINCS preparation",
    "files": {},
}

for region, path in ready_files.items():
    details = inspect_file(region, path)
    details["path"] = str(path)
    details["bytes"] = path.stat().st_size
    details["sha256"] = sha256(path)
    manifest["files"][region] = details

manifest_path = DEST_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Wrote:", manifest_path)
for region, info in manifest["files"].items():
    print(region, info["sha256"], info["path"])

## 7. Quick visual check

This displays the maximum SSH during the Ike window for each regional file.

In [ ]:
fig, axes = plt.subplots(1, len(ready_files), figsize=(12, 4), constrained_layout=True)
if len(ready_files) == 1:
    axes = [axes]

for ax, (region, path) in zip(axes, ready_files.items()):
    with xr.open_dataset(path) as ds:
        peak = ds["SSH_inst"].max("time").load()
    image = ax.pcolormesh(peak.xh, peak.yh, peak, shading="auto", cmap="viridis")
    fig.colorbar(image, ax=ax, label="maximum SSH (m)")
    ax.set_title(f"{region}: {REGIONS[region]}")
    ax.set_xlabel("longitude (degrees east)")
    ax.set_ylabel("latitude (degrees north)")
plt.show()

## 8. Paths to use in the SFINCS workflow

Use `ready_files['TX']` for the Texas/Galveston configuration and `ready_files['MS']` for the Mississippi configuration. These files contain gridded MOM6 SSH; use the appropriate downstream converter if the SFINCS model expects boundary-point forcing rather than gridded NetCDF.

In [ ]:
print("Texas MOM6 SSH source for SFINCS:")
print(ready_files["TX"])
print("\nMississippi MOM6 SSH source for SFINCS:")
print(ready_files["MS"])
print("\nManifest:")
print(manifest_path)